# Pratica 06 — Analisi del sentimento classica

**Corso:** NLP Classico con Python · **Lezione 06** · **Tempo stimato:** 60-75 minuti
**Notebook di lezione:** `notebooks_clase/clase06_sentimiento_clasico_IT.ipynb`

## Come funziona questa pratica

- Ogni esercizio ha una o più celle con `# TODO`: completale.
- Alla fine di ogni esercizio c'è un **controllo automatico** (`assert`). Se fallisce, il messaggio ti indica cosa rivedere.
- Gli esercizi da 1 a 4 usano un **lessico e delle frasi minuscoli** (cella di dati dopo il setup) così puoi calcolare tutto a mano; l'esercizio 5 e la sfida usano le recensioni e il lessico di `datos/` (in Colab, carica la cartella `datos/`). Sono testi sintetici scritti per il corso (*esempio illustrativo*).
- Le **soluzioni** sono alla fine del notebook, in una sezione separata. **Non aprirle prima di aver provato.**
- Esegui le celle in ordine. Librerie: `numpy`, `pandas`, `scipy` e `scikit-learn`.

---
## Cella 0 — Setup e dati

In [ ]:
import os
import re
import unicodedata

import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline

def cargar_datos(nombre):
    """Cerca un file di dati in datos/, ../datos/ e /content/datos/."""
    for base in ("datos", "../datos", "/content/datos"):
        ruta = os.path.join(base, nombre)
        if os.path.exists(ruta):
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/14-nlp-classico/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError(
        "Non trovo " + nombre + ". Carica la cartella datos/ del corso (in Colab: pannello dei file) o lasciala accanto al notebook.")

IDIOMAS = ("spa", "ita")
ARCHIVO_RESENAS = {"spa": "resenas_es.csv", "ita": "resenas_it.csv"}
ARCHIVO_LEXICO = {"spa": "lexico_polaridad_es.csv", "ita": "lexico_polaridad_it.csv"}
CLASES = ["negativo", "neutro", "positivo"]

def normalizar(texto):
    """Minuscolo e senza accenti: così il lessico funziona anche con il testo grezzo che ha perso gli accenti."""
    texto = unicodedata.normalize("NFD", texto.lower())
    return "".join(c for c in texto if unicodedata.category(c) != "Mn")

def tokenizar(texto):
    """Parole e segni di punteggiatura (i segni delimitano la finestra della negazione)."""
    return re.findall(r"\w+|[.,;:!?()]", normalizar(texto))

def cargar_lexico(idioma):
    tabla = pd.read_csv(cargar_datos(ARCHIVO_LEXICO[idioma]))
    return {normalizar(p): float(v) for p, v in zip(tabla["palabra"], tabla["polaridad"])}

def analizar(texto, lexico, idioma, ventana=3, negacion=True, intensificadores=True):
    """Restituisce (punteggio totale, lista dei contributi (parola, valore base, valore finale))."""
    tokens = tokenizar(texto)
    total, aportes = 0.0, []
    for i, palabra in enumerate(tokens):
        if palabra not in lexico:
            continue
        valor = lexico[palabra]
        multiplicador, invertir = 1.0, False
        for j in range(i - 1, max(i - ventana, 0) - 1, -1):    # all'indietro, fino a `ventana` token
            anterior = tokens[j]
            if not anterior[0].isalnum() or anterior in CORTES[idioma]:
                break                                          # punteggiatura o «ma»: fine della proposizione
            if negacion and anterior in NEGADORES[idioma]:
                invertir = True                                # più negatori di seguito invertono UNA sola volta
            elif intensificadores and anterior in INTENSIFICADORES[idioma]:
                multiplicador *= INTENSIFICADORES[idioma][anterior]
        final = valor * multiplicador * (-1 if invertir else 1)
        aportes.append((palabra, valor, final))
        total += final
    return total, aportes

def puntuar(texto, lexico, idioma, **opciones):
    return analizar(texto, lexico, idioma, **opciones)[0]


print("Librerie caricate correttamente.")

In [ ]:
# datos-es
LEXICO_MINI = {"spa": {"bueno": 1, "excelente": 2, "rápido": 1, "malo": -1, "horrible": -2, "lento": -1, "caro": -1},
               "ita": {"buono": 1, "eccellente": 2, "veloce": 1, "cattivo": -1, "orribile": -2, "lento": -1, "caro": -1}}
NEGADORES_P = {"spa": {"no", "nunca", "sin"}, "ita": {"non", "mai", "senza"}}
FRASES_P = {
    "spa": ["el servicio es bueno y rápido",
            "la comida es excelente",
            "el envío es lento y caro",
            "no es bueno",
            "nunca es lento",
            "el trato fue horrible",
            "el envío llegó ayer"],
    "ita": ["il servizio è buono e veloce",
            "il cibo è eccellente",
            "l'ordine è lento e caro",
            "non è buono",
            "mai lento",
            "il trattamento è stato orribile",
            "l'ordine è arrivato ieri"],
}
NEGADORES = {"spa": {"no", "ni", "nunca", "jamas", "tampoco", "sin", "nada", "ningun", "ninguno", "ninguna"},
             "ita": {"non", "mai", "senza", "nessun", "nessuno", "nessuna", "nemmeno", "neppure", "niente", "nulla"}}
INTENSIFICADORES = {"spa": {"muy": 1.5, "super": 1.5, "tremendamente": 1.5, "realmente": 1.5, "totalmente": 1.5,
                            "bastante": 1.25, "demasiado": 1.25, "poco": 0.5, "algo": 0.5},
                    "ita": {"molto": 1.5, "super": 1.5, "davvero": 1.5, "veramente": 1.5, "assolutamente": 1.5,
                            "estremamente": 1.5, "troppo": 1.25, "abbastanza": 1.25, "poco": 0.5, "alquanto": 0.75}}
CORTES = {"spa": {"pero", "aunque", "sino"}, "ita": {"ma", "pero", "sebbene", "tuttavia"}}   # le avversative tagliano la finestra

## Esercizio 1 — Un analizzatore lessicale minimo

Con `LEXICO_MINI` (un dizionario `parola → valore`), completa `puntuar_simple(frase, lexico)`: tokenizza con `\w+` in minuscolo e **somma il valore di ogni parola presente nel lessico** (quelle assenti valgono 0).

Prima di programmare, calcola a mano i punteggi delle sette frasi di `FRASES_P["ita"]`: `2, 2, −2, 1, −1, −2, 0`. Nota che «non è buono» **dà +1**: non trattiamo ancora la negazione.

In [ ]:
def puntuar_simple(frase, lexico):
    tokens = re.findall(r"\w+", frase.lower())
    total = None        # TODO: somma dei valori delle parole presenti nel lessico
    return total

puntos_1 = {idioma: [puntuar_simple(f, LEXICO_MINI[idioma]) for f in FRASES_P[idioma]] for idioma in IDIOMAS}

# Controllo automatico
assert all(p is not None for idioma in IDIOMAS for p in puntos_1[idioma]), "Completa le righe contrassegnate con TODO"
for idioma in IDIOMAS:
    assert puntos_1[idioma] == [2, 2, -2, 1, -1, -2, 0], "i punteggi devono essere 2, 2, −2, 1, −1, −2, 0"
print("Esercizio 1 corretto ✔")

## Esercizio 2 — Negazione con finestra

Completa `puntuar_neg(frase, lexico, negadores, ventana)`: come prima, ma se **uno qualsiasi dei `ventana` token precedenti** a una parola del lessico è un negatore, **inverti il suo segno** (una volta sola anche se ci sono più negatori).

Calcola a mano: con `ventana=2`, «non è buono» vale −1 e «mai lento» vale +1. E «non è un servizio buono» vale **+1** con `ventana=2` (il `non` resta a quattro token da `buono`) ma **−1** con `ventana=4`: la finestra ha un prezzo.

In [ ]:
def puntuar_neg(frase, lexico, negadores, ventana):
    tokens = re.findall(r"\w+", frase.lower())
    total = 0
    for i, t in enumerate(tokens):
        if t in lexico:
            valor = lexico[t]
            anteriores = tokens[max(0, i - ventana):i]
            # TODO: se qualche token di `anteriores` è in `negadores`, inverti il segno di `valor`
            total += valor
    return total

FRASE_LEJOS = {"spa": "no es un servicio bueno", "ita": "non è un servizio buono"}   # dato-es
res_2 = {idioma: ([puntuar_neg(f, LEXICO_MINI[idioma], NEGADORES_P[idioma], 2) for f in FRASES_P[idioma]],
                  puntuar_neg(FRASE_LEJOS[idioma], LEXICO_MINI[idioma], NEGADORES_P[idioma], 2),
                  puntuar_neg(FRASE_LEJOS[idioma], LEXICO_MINI[idioma], NEGADORES_P[idioma], 4)) for idioma in IDIOMAS}

# Controllo automatico
for idioma in IDIOMAS:
    lista, cerca, lejos = res_2[idioma]
    assert lista == [2, 2, -2, -1, 1, -2, 0], "con ventana=2 i punteggi devono essere 2, 2, −2, −1, 1, −2, 0"
    assert cerca == 1 and lejos == -1, "con ventana=2 la frase lontana vale +1 e con ventana=4 vale −1"
print("Esercizio 2 corretto ✔")

## Esercizio 3 — Dal punteggio alla classe, accuratezza e soglia

Completa tre funzioni:

- `clasificar(p, umbral)`: `"positivo"` se `p > umbral`, `"negativo"` se `p < -umbral`, `"neutro"` altrimenti;
- `exactitud(y, pred)`: frazione di risposte corrette;
- `elegir_umbral(puntos, y, candidatos)`: la soglia tra `candidatos` con la maggiore accuratezza (a parità, la **minore**).

Con gli otto punteggi di `PUNTOS` e le loro etichette reali `ETIQUETAS`, calcola a mano l'accuratezza con `umbral=0` (**6 su 8 = 0,75**), con `umbral=1` (anche 6 su 8) e con `umbral=2` (4 su 8).

In [ ]:
PUNTOS = [2, -1, 0, 1, -2, 0, 3, -1]
ETIQUETAS = ["positivo", "negativo", "neutro", "neutro", "negativo", "neutro", "positivo", "positivo"]

def clasificar(p, umbral):
    # TODO: restituisci "positivo", "negativo" o "neutro"
    return None

def exactitud(y, pred):
    # TODO: frazione di posizioni in cui y e pred coincidono
    return None

def elegir_umbral(puntos, y, candidatos):
    # TODO: prova ogni candidato e restituisci quello con la maggiore accuratezza (il minore in caso di parità)
    return None

# Controllo automatico
assert clasificar(0, 0) is not None and exactitud(["a"], ["a"]) is not None and elegir_umbral(PUNTOS, ETIQUETAS, (0, 1)) is not None, "Completa le righe contrassegnate con TODO"
assert [clasificar(p, 0) for p in (2, -1, 0)] == ["positivo", "negativo", "neutro"], "con soglia 0: 2 → positivo, −1 → negativo, 0 → neutro"
assert clasificar(1, 1) == "neutro" and clasificar(-1, 1) == "neutro", "la soglia è stretta: p = soglia è neutro"
assert exactitud(ETIQUETAS, [clasificar(p, 0) for p in PUNTOS]) == 0.75
assert exactitud(ETIQUETAS, [clasificar(p, 1) for p in PUNTOS]) == 0.75
assert exactitud(ETIQUETAS, [clasificar(p, 2) for p in PUNTOS]) == 0.5
assert elegir_umbral(PUNTOS, ETIQUETAS, (2, 1, 0)) == 0, "0 e 1 pareggiano con 0,75: vince il minore"
print("Esercizio 3 corretto ✔")

## Esercizio 4 — Errori per categorie

Una sola percentuale nasconde dove sbaglia il metodo. Con la tabella `TABLA_ERR` (dieci recensioni, con la categoria `A` = senza parole del lessico, `B` = con negazione e `C` = polarità chiara), completa:

- `tasa_error_por_categoria(tabla)`: una `Series` con il **tasso di errore** (frazione di righe con `etiqueta != pred`) di ciascuna categoria;
- `matriz_confusion(tabla)`: una tabella che incrocia `etiqueta` (righe) e `pred` (colonne) con `pd.crosstab`.

Calcola a mano: `A` sbaglia 2 su 4 (0,50), `B` sbaglia 1 su 2 (0,50) e `C` sbaglia 1 su 4 (0,25).

In [ ]:
TABLA_ERR = pd.DataFrame({
    "categoria": ["A", "A", "A", "A", "B", "B", "C", "C", "C", "C"],
    "etiqueta": ["negativo", "positivo", "neutro", "neutro", "negativo", "positivo", "positivo", "positivo", "negativo", "neutro"],
    "pred": ["neutro", "neutro", "neutro", "neutro", "negativo", "negativo", "positivo", "positivo", "negativo", "positivo"],
})

def tasa_error_por_categoria(tabla):
    # TODO: raggruppa per categoria e calcola la media di (etiqueta != pred)
    return None

def matriz_confusion(tabla):
    # TODO: pd.crosstab di etiqueta (righe) e pred (colonne)
    return None

# Controllo automatico
tasas = tasa_error_por_categoria(TABLA_ERR)
matriz = matriz_confusion(TABLA_ERR)
assert tasas is not None and matriz is not None, "Completa le righe contrassegnate con TODO"
assert dict(tasas.round(2)) == {"A": 0.5, "B": 0.5, "C": 0.25}, "i tassi di errore devono essere A 0,50, B 0,50 e C 0,25"
assert int(matriz.to_numpy().sum()) == 10 and int(matriz.loc["neutro", "neutro"]) == 2, "la matrice deve sommare 10 e avere 2 neutri classificati bene"
assert int(np.trace(matriz.reindex(index=CLASES, columns=CLASES, fill_value=0).to_numpy())) == 6, "ci sono 6 risposte corrette sulla diagonale"
print("Esercizio 4 corretto ✔")

## Esercizio 5 — Il classificatore supervisionato sulla stessa partizione

Con le recensioni reali di `datos/` (scegli `IDIOMA`), completa:

- `particion(df)`: una **sola** partizione 70/30 stratificata per `etiqueta` con `random_state=42` (restituisce `entren, prueba`);
- `entrenar_y_medir(entren, prueba)`: addestra una `Pipeline` di `TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)` e `LogisticRegression(max_iter=1000)` con il testo **grezzo** (`texto`) dell'addestramento, e restituisce `(modelo, accuratezza sul test)`.

Nota l'ordine: il vettorizzatore si adatta solo con l'addestramento (lezione 4). Con 300 recensioni la partizione ha 210 e 90 righe.

In [ ]:
IDIOMA = "spa"            # o "ita"
df_res = pd.read_csv(cargar_datos(ARCHIVO_RESENAS[IDIOMA]))

def particion(df):
    entren, prueba = None, None      # TODO: train_test_split stratificato 70/30 con random_state=42
    return entren, prueba

def entrenar_y_medir(entren, prueba):
    modelo = None                    # TODO: Pipeline TF-IDF (1-2 grammi, sublinear_tf) + regressione logistica
    # TODO: adatta il modello con entren["texto"] e entren["etiqueta"]
    acierto = None                   # TODO: accuratezza su prueba["texto"]
    return modelo, acierto

entren, prueba = particion(df_res)
modelo, acierto_lr = entrenar_y_medir(entren, prueba)

# Controllo automatico
assert entren is not None and prueba is not None and modelo is not None and acierto_lr is not None, "Completa le righe contrassegnate con TODO"
assert len(entren) == 210 and len(prueba) == 90, "la partizione 70/30 di 300 recensioni ha 210 e 90 righe"
assert set(entren["id"]).isdisjoint(set(prueba["id"])), "addestramento e test non devono condividere recensioni"
assert prueba["etiqueta"].value_counts().to_dict() == {"negativo": 30, "neutro": 30, "positivo": 30}, "la partizione deve essere stratificata: 30 recensioni per classe nel test"
assert isinstance(modelo.steps[0][1], TfidfVectorizer) and isinstance(modelo.steps[-1][1], LogisticRegression), "la Pipeline deve essere TfidfVectorizer + LogisticRegression"
assert modelo.steps[0][1].ngram_range == (1, 2) and modelo.steps[0][1].sublinear_tf, "usa ngram_range=(1, 2) e sublinear_tf=True"
assert 0.5 < acierto_lr <= 1.0, "l'accuratezza dovrebbe stare nettamente sopra il caso (1/3)"
assert abs(acierto_lr - modelo.score(prueba["texto"], prueba["etiqueta"])) < 1e-12
print(f"Esercizio 5 corretto ✔ (accuratezza sul test: {acierto_lr:.3f}; cifra illustrativa)")

---
## Sfida finale — Il lessico aiuta il modello supervisionato?

Con la partizione dell'esercizio 5 (`entren`, `prueba`) e il **lessico reale del corso** (`LEXICO_REAL`, già caricato, con `analizar` della lezione), costruisci un modello **ibrido**: le colonne di TF-IDF **più una colonna extra** con il punteggio del lessico di ogni recensione.

1. Scrivi `puntuacion_lexico(textos)`: array di `float` con `puntuar(texto, LEXICO_REAL[IDIOMA], IDIOMA)` per ogni testo.
2. Scrivi `medir_hibrido()`: addestra TF-IDF (1-2 grammi, `sublinear_tf`) con `entren["texto"]`, aggiungi la colonna del lessico con `scipy.sparse.hstack` (adatta il vettorizzatore **solo** con l'addestramento), addestra `LogisticRegression(max_iter=1000)` e restituisce un dizionario con le accuratezze sullo **stesso test**: `{"solo_tfidf": ..., "hibrido": ...}`.

Non si chiede che l'ibrido vinca: **misura e descrivi** ciò che esce. Con 90 recensioni di test, una differenza di pochi punti è rumore.

In [ ]:
LEXICO_REAL = {idioma: cargar_lexico(idioma) for idioma in IDIOMAS}

def puntuacion_lexico(textos):
    # TODO: array di float con il punteggio del lessico di ogni testo
    return None

def medir_hibrido():
    # TODO: TF-IDF adattato con entren + colonna del lessico (sp.hstack) + regressione logistica; restituisci il dizionario
    return None

resultado = medir_hibrido()

# Controllo automatico
assert resultado is not None, "Completa le righe contrassegnate con TODO"
assert set(resultado) == {"solo_tfidf", "hibrido"}, "il dizionario deve avere le chiavi solo_tfidf e hibrido"
assert all(0.5 < v <= 1.0 for v in resultado.values()), "le due accuratezze devono essere nettamente sopra il caso"
assert abs(resultado["solo_tfidf"] - acierto_lr) < 1e-12, "«solo_tfidf» deve coincidere con l'accuratezza dell'esercizio 5 (stessa partizione e stesso modello)"
ejemplo = puntuacion_lexico(prueba["texto"].head(5))
assert len(ejemplo) == 5 and ejemplo[0] == puntuar(prueba["texto"].iloc[0], LEXICO_REAL[IDIOMA], IDIOMA), "puntuacion_lexico deve usare puntuar con il lessico reale"
print(f"Sfida finale corretta ✔ (solo TF-IDF {resultado['solo_tfidf']:.3f} · ibrido {resultado['hibrido']:.3f}; cifre illustrative)")

**Le tue conclusioni (scrivi 3-4 frasi):** nella sfida, l'ibrido migliora, peggiora o resta uguale al solo TF-IDF? Di quante recensioni di test è la differenza? Cambieresti conclusione con un'altra partizione? Cita una categoria di errore (negazione, ironia, senza parole del lessico) in cui pensi che il lessico aiuterebbe o no.

*(La tua risposta qui.)*

---
## Soluzioni (non guardare prima di aver provato!)

Ogni cella seguente contiene la soluzione di un esercizio. Puoi eseguirle per confrontarle con il tuo risultato.

### Soluzione dell'esercizio 1

In [ ]:
def puntuar_simple(frase, lexico):
    tokens = re.findall(r"\w+", frase.lower())
    total = sum(lexico.get(t, 0) for t in tokens)
    return total

puntos_1 = {idioma: [puntuar_simple(f, LEXICO_MINI[idioma]) for f in FRASES_P[idioma]] for idioma in IDIOMAS}

# Controllo automatico
assert all(p is not None for idioma in IDIOMAS for p in puntos_1[idioma]), "Completa le righe contrassegnate con TODO"
for idioma in IDIOMAS:
    assert puntos_1[idioma] == [2, 2, -2, 1, -1, -2, 0], "i punteggi devono essere 2, 2, −2, 1, −1, −2, 0"
print("Esercizio 1 corretto ✔")

### Soluzione dell'esercizio 2

In [ ]:
def puntuar_neg(frase, lexico, negadores, ventana):
    tokens = re.findall(r"\w+", frase.lower())
    total = 0
    for i, t in enumerate(tokens):
        if t in lexico:
            valor = lexico[t]
            anteriores = tokens[max(0, i - ventana):i]
            if any(a in negadores for a in anteriores):
                valor = -valor
            total += valor
    return total

FRASE_LEJOS = {"spa": "no es un servicio bueno", "ita": "non è un servizio buono"}   # dato-es
res_2 = {idioma: ([puntuar_neg(f, LEXICO_MINI[idioma], NEGADORES_P[idioma], 2) for f in FRASES_P[idioma]],
                  puntuar_neg(FRASE_LEJOS[idioma], LEXICO_MINI[idioma], NEGADORES_P[idioma], 2),
                  puntuar_neg(FRASE_LEJOS[idioma], LEXICO_MINI[idioma], NEGADORES_P[idioma], 4)) for idioma in IDIOMAS}

# Controllo automatico
for idioma in IDIOMAS:
    lista, cerca, lejos = res_2[idioma]
    assert lista == [2, 2, -2, -1, 1, -2, 0], "con ventana=2 i punteggi devono essere 2, 2, −2, −1, 1, −2, 0"
    assert cerca == 1 and lejos == -1, "con ventana=2 la frase lontana vale +1 e con ventana=4 vale −1"
print("Esercizio 2 corretto ✔")

### Soluzione dell'esercizio 3

In [ ]:
PUNTOS = [2, -1, 0, 1, -2, 0, 3, -1]
ETIQUETAS = ["positivo", "negativo", "neutro", "neutro", "negativo", "neutro", "positivo", "positivo"]

def clasificar(p, umbral):
    if p > umbral:
        return "positivo"
    if p < -umbral:
        return "negativo"
    return "neutro"

def exactitud(y, pred):
    return sum(a == b for a, b in zip(y, pred)) / len(y)

def elegir_umbral(puntos, y, candidatos):
    mejor, mejor_acierto = None, -1
    for u in sorted(candidatos):
        acierto = exactitud(y, [clasificar(p, u) for p in puntos])
        if acierto > mejor_acierto:
            mejor, mejor_acierto = u, acierto
    return mejor

# Controllo automatico
assert clasificar(0, 0) is not None and exactitud(["a"], ["a"]) is not None and elegir_umbral(PUNTOS, ETIQUETAS, (0, 1)) is not None, "Completa le righe contrassegnate con TODO"
assert [clasificar(p, 0) for p in (2, -1, 0)] == ["positivo", "negativo", "neutro"], "con soglia 0: 2 → positivo, −1 → negativo, 0 → neutro"
assert clasificar(1, 1) == "neutro" and clasificar(-1, 1) == "neutro", "la soglia è stretta: p = soglia è neutro"
assert exactitud(ETIQUETAS, [clasificar(p, 0) for p in PUNTOS]) == 0.75
assert exactitud(ETIQUETAS, [clasificar(p, 1) for p in PUNTOS]) == 0.75
assert exactitud(ETIQUETAS, [clasificar(p, 2) for p in PUNTOS]) == 0.5
assert elegir_umbral(PUNTOS, ETIQUETAS, (2, 1, 0)) == 0, "0 e 1 pareggiano con 0,75: vince il minore"
print("Esercizio 3 corretto ✔")

### Soluzione dell'esercizio 4

In [ ]:
TABLA_ERR = pd.DataFrame({
    "categoria": ["A", "A", "A", "A", "B", "B", "C", "C", "C", "C"],
    "etiqueta": ["negativo", "positivo", "neutro", "neutro", "negativo", "positivo", "positivo", "positivo", "negativo", "neutro"],
    "pred": ["neutro", "neutro", "neutro", "neutro", "negativo", "negativo", "positivo", "positivo", "negativo", "positivo"],
})

def tasa_error_por_categoria(tabla):
    return (tabla["etiqueta"] != tabla["pred"]).groupby(tabla["categoria"]).mean()

def matriz_confusion(tabla):
    return pd.crosstab(tabla["etiqueta"], tabla["pred"])

# Controllo automatico
tasas = tasa_error_por_categoria(TABLA_ERR)
matriz = matriz_confusion(TABLA_ERR)
assert tasas is not None and matriz is not None, "Completa le righe contrassegnate con TODO"
assert dict(tasas.round(2)) == {"A": 0.5, "B": 0.5, "C": 0.25}, "i tassi di errore devono essere A 0,50, B 0,50 e C 0,25"
assert int(matriz.to_numpy().sum()) == 10 and int(matriz.loc["neutro", "neutro"]) == 2, "la matrice deve sommare 10 e avere 2 neutri classificati bene"
assert int(np.trace(matriz.reindex(index=CLASES, columns=CLASES, fill_value=0).to_numpy())) == 6, "ci sono 6 risposte corrette sulla diagonale"
print("Esercizio 4 corretto ✔")

### Soluzione dell'esercizio 5

In [ ]:
IDIOMA = "spa"
df_res = pd.read_csv(cargar_datos(ARCHIVO_RESENAS[IDIOMA]))

def particion(df):
    entren, prueba = train_test_split(df, test_size=0.3, stratify=df["etiqueta"], random_state=42)
    return entren, prueba

def entrenar_y_medir(entren, prueba):
    modelo = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True), LogisticRegression(max_iter=1000))
    modelo.fit(entren["texto"], entren["etiqueta"])
    acierto = accuracy_score(prueba["etiqueta"], modelo.predict(prueba["texto"]))
    return modelo, acierto

entren, prueba = particion(df_res)
modelo, acierto_lr = entrenar_y_medir(entren, prueba)

# Controllo automatico
assert entren is not None and prueba is not None and modelo is not None and acierto_lr is not None, "Completa le righe contrassegnate con TODO"
assert len(entren) == 210 and len(prueba) == 90, "la partizione 70/30 di 300 recensioni ha 210 e 90 righe"
assert set(entren["id"]).isdisjoint(set(prueba["id"])), "addestramento e test non devono condividere recensioni"
assert prueba["etiqueta"].value_counts().to_dict() == {"negativo": 30, "neutro": 30, "positivo": 30}, "la partizione deve essere stratificata: 30 recensioni per classe nel test"
assert isinstance(modelo.steps[0][1], TfidfVectorizer) and isinstance(modelo.steps[-1][1], LogisticRegression), "la Pipeline deve essere TfidfVectorizer + LogisticRegression"
assert modelo.steps[0][1].ngram_range == (1, 2) and modelo.steps[0][1].sublinear_tf, "usa ngram_range=(1, 2) e sublinear_tf=True"
assert 0.5 < acierto_lr <= 1.0, "l'accuratezza dovrebbe stare nettamente sopra il caso (1/3)"
assert abs(acierto_lr - modelo.score(prueba["texto"], prueba["etiqueta"])) < 1e-12
print(f"Esercizio 5 corretto ✔ (accuratezza sul test: {acierto_lr:.3f}; cifra illustrativa)")

### Soluzione della sfida finale

In [ ]:
LEXICO_REAL = {idioma: cargar_lexico(idioma) for idioma in IDIOMAS}

def puntuacion_lexico(textos):
    return np.array([puntuar(t, LEXICO_REAL[IDIOMA], IDIOMA) for t in textos], dtype=float)

def medir_hibrido():
    vec = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
    X_tr = vec.fit_transform(entren["texto"])
    X_te = vec.transform(prueba["texto"])
    solo = LogisticRegression(max_iter=1000).fit(X_tr, entren["etiqueta"])
    H_tr = sp.hstack([X_tr, sp.csr_matrix(puntuacion_lexico(entren["texto"]).reshape(-1, 1))]).tocsr()
    H_te = sp.hstack([X_te, sp.csr_matrix(puntuacion_lexico(prueba["texto"]).reshape(-1, 1))]).tocsr()
    hibrido = LogisticRegression(max_iter=1000).fit(H_tr, entren["etiqueta"])
    return {"solo_tfidf": accuracy_score(prueba["etiqueta"], solo.predict(X_te)),
            "hibrido": accuracy_score(prueba["etiqueta"], hibrido.predict(H_te))}

resultado = medir_hibrido()

# Controllo automatico
assert resultado is not None, "Completa le righe contrassegnate con TODO"
assert set(resultado) == {"solo_tfidf", "hibrido"}, "il dizionario deve avere le chiavi solo_tfidf e hibrido"
assert all(0.5 < v <= 1.0 for v in resultado.values()), "le due accuratezze devono essere nettamente sopra il caso"
assert abs(resultado["solo_tfidf"] - acierto_lr) < 1e-12, "«solo_tfidf» deve coincidere con l'accuratezza dell'esercizio 5 (stessa partizione e stesso modello)"
ejemplo = puntuacion_lexico(prueba["texto"].head(5))
assert len(ejemplo) == 5 and ejemplo[0] == puntuar(prueba["texto"].iloc[0], LEXICO_REAL[IDIOMA], IDIOMA), "puntuacion_lexico deve usare puntuar con il lessico reale"
print(f"Sfida finale corretta ✔ (solo TF-IDF {resultado['solo_tfidf']:.3f} · ibrido {resultado['hibrido']:.3f}; cifre illustrative)")

---
## Autovalutazione della lezione

Rispondi con onestà (una o due frasi ciascuna):

1. Quale esercizio ti è costato di più e perché?
2. Quale concetto della lezione non ti è rimasto chiaro (lessico di polarità, finestra di negazione, soglia, confronto sulla stessa partizione, categorie di errore, ironia)?
3. Cosa cambieresti o ripeteresti la prossima volta?
4. Quanto tempo hai dedicato in totale?

*(Le tue risposte qui.)*